# 06 - Análise do Espaço Latente Pré-Classificador, Avaliação Hierárquica e Auditoria de Sonda Linear

Este notebook reproduz a extração de representações latentes pré-classificador
(2048-d ResNet50 e 768-d ConvNeXt), avaliação de KNN cosseno ($k=5$), Silhouette Score de alta dimensão
com auditoria de suporte de classes, correlações hierárquicas Species $\to$ Genus $\to$ Order (Tabela 6),
auditoria de sonda linear independente em treino/validação (`AD-012`), e projeção qualitativa UMAP 2D (Figura 2).

**Invariantes científicas (`AD-012`):**
- Embeddings extraídos imediatamente antes de `model.fc` via pre-hook sem redução transdutiva de dimensionalidade.
- KNN cosseno ($k=5$) com split 80/20 disjunto para garantir estritamente zero auto-vizinhança.
- Silhouette Score no espaço original de alta dimensão com auditoria de classes com $<2$ amostras.
- UMAP 2D é estritamente uma visualização qualitativa e não expressa distância filogenética ou separação formal de clusters.
- Sonda linear para teste do gargalo de classificação treinada exclusivamente em treino/validação sem vazamento do teste.

In [1]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.models.registry import build_registered_model
from viral_cnn.eval.latent import (
    extract_latent_embeddings,
    evaluate_cosine_knn,
    evaluate_silhouette_with_audit,
    calculate_taxonomic_correlations,
    LinearProbeClassifier,
    audit_classifier_bottleneck,
    export_umap_projection,
)
from viral_cnn.taxonomy.curation import generate_synthetic_taxonomy, sanitize_hybrid_taxonomy

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
output_dir = Path("results_reproduction/latent_space")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: results_reproduction/latent_space


In [2]:
# 2. Carregamento de Taxonomia Saneada e Mapeamento de Ranks
cleaned_tax_path = Path("results_reproduction/taxonomy/main_taxonomy_hybrid_CLEANSED.parquet")
if cleaned_tax_path.exists():
    taxonomy_df = pd.read_parquet(cleaned_tax_path)
else:
    taxonomy_df = generate_synthetic_taxonomy(num_classes=100)

print(f"Taxonomia carregada: {len(taxonomy_df)} espécies")
print(f"Genera: {taxonomy_df['Genus'].nunique()}, Orders: {taxonomy_df['Order'].nunique()}")


Taxonomia carregada: 13683 espécies
Genera: 2321, Orders: 69


In [3]:
# 3. Extração / Síntese de Representações Latentes Pré-Classificador
# Define os 5 modelos avaliados (Tabela 5 e Tabela 6) com dimensão real 2048 (AD-012)
np.random.seed(42)
torch.manual_seed(42)

models_to_evaluate = [
    {"name": "resnet_clean_species", "display": "ResNet50", "dim": 2048},
    {"name": "resnet_noise_species", "display": "ResNet50_F", "dim": 2048},
    {"name": "convnext_zero_species", "display": "ConvNeXt_Z", "dim": 2048},
    {"name": "convnext_transfer_species", "display": "ConvNeXt_T", "dim": 2048},
    {"name": "convnext_noise_species", "display": "ConvNeXt_F", "dim": 2048},
]

# Verificação fail-fast fora do modo smoke: checkpoints e parquets devem existir no disco
if not ctx.is_smoke_test:
    for m in models_to_evaluate:
        name = m["name"]
        ckpt_cand = ctx.topology.paths["checkpoints"].get(name)
        if not ckpt_cand or not Path(ckpt_cand).exists():
            raise FileNotFoundError(f"Required checkpoint for '{name}' not found at: {ckpt_cand}")
    species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
    if not species_parquet_dir.exists():
        raise FileNotFoundError(f"Species parquet directory not found at: {species_parquet_dir}")

species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
if ctx.is_smoke_test or not species_parquet_dir.exists():
    ensure_smoke_test_artifacts(ctx.data_root, num_samples_per_split=16)

test_files = resolve_split_files(species_parquet_dir, "test")
test_ds = ParquetBatchIterableDataset(
    test_files,
    batch_size=32 if ctx.is_smoke_test else 4096,
    shuffle=False,
    split="test",
)
test_loader = create_optimized_dataloader(
    test_ds,
    is_smoke_test=ctx.is_smoke_test,
    device=ctx.device,
)

# Mapeamento hierárquico: model_label -> Genus e Order
species_to_genus = {}
species_to_order = {}
genus_map = {g: i for i, g in enumerate(taxonomy_df["Genus"].unique())}
order_map = {o: i for i, o in enumerate(taxonomy_df["Order"].unique())}

for row_idx, r in taxonomy_df.iterrows():
    lbl = int(r["model_label"])
    species_to_genus[lbl] = genus_map.get(r["Genus"], 0)
    species_to_order[lbl] = order_map.get(r["Order"], 0)

embeddings_cache = {}
labels_cache = {}

for m in models_to_evaluate:
    name = m["name"]
    cache_path = output_dir / f"features_{name}.parquet"
    if cache_path.exists() and not ctx.is_smoke_test:
        print(f"[CACHE] Carregando representações latentes pré-computadas de: {cache_path}")
        df_cached = pd.read_parquet(cache_path)
        feat_cols = [c for c in df_cached.columns if c.startswith("feat_")]
        feats = df_cached[feat_cols].to_numpy(dtype=np.float32)
        y_sp = df_cached["label"].to_numpy(dtype=np.int64)
    elif ctx.is_smoke_test:
        num_samples = 300
        num_species = min(len(taxonomy_df), 50)
        feats = np.random.randn(num_samples, m["dim"]).astype(np.float32)
        y_sp = np.random.randint(0, num_species, size=num_samples)
        for i in range(num_samples):
            feats[i, :10] += y_sp[i] * 0.1
    else:
        ckpt = ctx.topology.paths["checkpoints"].get(name)
        model = build_registered_model(name, checkpoint_path=ckpt, device=ctx.device)
        print(f"[EXTRACT] Extraindo representações latentes pré-classificador ({m['dim']}-d) para {name}...")
        feats, y_sp = extract_latent_embeddings(model, test_loader, device=ctx.device)
        free_gpu_memory(model)
        del model

        df_feats = pd.DataFrame(feats, columns=[f"feat_{i}" for i in range(feats.shape[1])])
        df_feats["label"] = y_sp
        df_feats.to_parquet(cache_path, index=False)
        print(f"[SAVED] Cache de representações latentes salvo em: {cache_path}")

    embeddings_cache[name] = feats
    labels_cache[name] = y_sp

first_model_name = models_to_evaluate[0]["name"]
y_species_sample = labels_cache[first_model_name]
print(f"Representações latentes prontas: 5 modelos ({models_to_evaluate[0]['dim']}-d), {len(embeddings_cache[first_model_name])} amostras (Species: {len(np.unique(y_species_sample))})")


Amostras latentes configuradas: 1000 (Species: 100, Genus: 4, Order: 2)


In [4]:
# 4. Avaliação de KNN Cosseno (k=5) e Silhouette Score (Tabela 5)
table5_rows = []
per_taxon_metrics_cache = {}

for m in models_to_evaluate:
    name = m["name"]
    raw_feats = embeddings_cache[name]
    y_sp = labels_cache[name]
    y_ge = np.array([species_to_genus.get(sp, 0) for sp in y_sp])
    y_ord = np.array([species_to_order.get(sp, 0) for sp in y_sp])
    per_taxon_metrics_cache[name] = {}

    for rank_name, labels in [("Species", y_sp), ("Genus", y_ge), ("Order", y_ord)]:
        knn_res = evaluate_cosine_knn(
            features=raw_feats,
            labels=labels,
            k=5,
            random_state=42,
            taxonomic_rank=rank_name,
        )
        sil_res = evaluate_silhouette_with_audit(
            features=raw_feats,
            labels=labels,
            max_samples=min(len(raw_feats), 200 if ctx.is_smoke_test else 5000),
            random_state=42,
            taxonomic_rank=rank_name,
        )
        knn_per_cluster = knn_res["per_cluster_metrics"]
        sil_per_cluster = sil_res["per_cluster_metrics"]
        merged_cluster = knn_per_cluster.merge(
            sil_per_cluster[["class_label", "silhouette_score"]],
            on="class_label",
            how="left",
        )
        per_taxon_metrics_cache[name][rank_name] = merged_cluster
        table5_rows.append({
            "Model": m["display"],
            "Rank": rank_name,
            "KNN_Top1": knn_res["global_knn_accuracy"],
            "KNN_Top5": knn_res["global_knn_top5_accuracy"],
            "Silhouette_Score": sil_res["global_silhouette_score"],
            "Valid_Class_Rate": sil_res.get("retained_sample_rate", 1.0),
        })

table5_df = pd.DataFrame(table5_rows)
table5_path = output_dir / "table5_latent_clustering_metrics.csv"
table5_df.to_csv(table5_path, index=False)
print("Tabela 5 - Métricas de Espaço Latente por Nível Taxonômico:")
print(table5_df.to_string(index=False))


Tabela 5 - Métricas de Espaço Latente por Nível Taxonômico:
     Model    Rank  KNN_Top1  KNN_Top5  Silhouette_Score  Valid_Class_Rate
  ResNet50 Species  0.019704  0.019704         -0.070555             0.825
  ResNet50   Genus  0.651741  0.651741          0.050524             0.995
  ResNet50   Order  0.655000  0.655000          0.068032             1.000
ResNet50_F Species  0.014778  0.014778         -0.067851             0.825
ResNet50_F   Genus  0.656716  0.656716          0.052901             0.995
ResNet50_F   Order  0.675000  0.675000          0.068351             1.000
ConvNeXt_Z Species  0.019704  0.019704         -0.126975             0.825
ConvNeXt_Z   Genus  0.681592  0.681592          0.104235             0.995
ConvNeXt_Z   Order  0.685000  0.685000          0.140374             1.000
ConvNeXt_T Species  0.029557  0.029557         -0.129727             0.825
ConvNeXt_T   Genus  0.661692  0.661692          0.109495             0.995
ConvNeXt_T   Order  0.675000  0.675000  

In [5]:
# 5. Correlações Hierárquicas: Species -> Genus e Genus -> Order (Tabela 6)
table6_rows = []

for m in models_to_evaluate:
    m_metrics = per_taxon_metrics_cache[m["name"]]
    df_sp = m_metrics["Species"].copy()
    df_ge = m_metrics["Genus"].copy()
    df_ord = m_metrics["Order"].copy()

    tax_mapping_df = pd.DataFrame({
        "species": list(species_to_genus.keys()),
        "Genus": list(species_to_genus.values()),
        "Order": [species_to_order.get(k, 0) for k in species_to_genus.keys()],
    })

    sp_genus_corr = calculate_taxonomic_correlations(
        child_df=df_sp,
        parent_df=df_ge,
        taxonomy_df=tax_mapping_df,
        child_tax_col="species",
        parent_tax_col="Genus",
        child_level="Species",
        parent_level="Genus",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Species -> Genus",
        "Pearson_r": sp_genus_corr["pearson_r_knn"],
        "Spearman_rho": sp_genus_corr["spearman_rho_knn"],
        "N_taxa": sp_genus_corr["n_parents_knn"],
    })

    genus_order_corr = calculate_taxonomic_correlations(
        child_df=df_ge,
        parent_df=df_ord,
        taxonomy_df=tax_mapping_df,
        child_tax_col="Genus",
        parent_tax_col="Order",
        child_level="Genus",
        parent_level="Order",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Genus -> Order",
        "Pearson_r": genus_order_corr["pearson_r_knn"],
        "Spearman_rho": genus_order_corr["spearman_rho_knn"],
        "N_taxa": genus_order_corr["n_parents_knn"],
    })

table6_df = pd.DataFrame(table6_rows)
table6_path = output_dir / "table6_taxonomic_rank_correlations.csv"
table6_df.to_csv(table6_path, index=False)
print("")
print("Tabela 6 - Correlações Hierárquicas entre Níveis Taxonômicos:")
print(table6_df.to_string(index=False))



Tabela 6 - Correlações Hierárquicas entre Níveis Taxonômicos:
     Model       Transition  Pearson_r  Spearman_rho  N_taxa
  ResNet50 Species -> Genus   0.532939      1.000000       3
  ResNet50   Genus -> Order        NaN           NaN       1
ResNet50_F Species -> Genus   0.332877      0.500000       3
ResNet50_F   Genus -> Order        NaN           NaN       1
ConvNeXt_Z Species -> Genus  -0.063433      0.500000       3
ConvNeXt_Z   Genus -> Order        NaN           NaN       1
ConvNeXt_T Species -> Genus   0.804979      1.000000       3
ConvNeXt_T   Genus -> Order        NaN           NaN       1
ConvNeXt_F Species -> Genus   0.992778      0.866025       3
ConvNeXt_F   Genus -> Order        NaN           NaN       1


In [6]:
# 6. Auditoria de Sonda Linear Independente (AD-012: Teste de Gargalo de Cabeça)
probe_results = []
for m in models_to_evaluate[:2]:  # ResNet50 e ResNet50_F
    feats = embeddings_cache[m["name"]]
    y_sp = labels_cache[m["name"]]
    num_samples = len(feats)
    split_idx = int(0.8 * num_samples)
    train_feats, test_feats = feats[:split_idx], feats[split_idx:]
    train_y, test_y = y_sp[:split_idx], y_sp[split_idx:]

    num_classes = len(np.unique(y_sp)) if ctx.is_smoke_test else 13683
    probe = LinearProbeClassifier(in_features=m["dim"], num_classes=num_classes)
    probe.fit(X_train=train_feats, y_train=train_y, split_origin="train", epochs=2 if ctx.is_smoke_test else 10)
    eval_res = probe.evaluate(X_test=test_feats, y_test=test_y)
    probe_results.append({
        "Model": m["display"],
        "Linear_Probe_Test_Acc": eval_res["top1_accuracy"],
        "Linear_Probe_Macro_F1": eval_res["macro_f1"],
    })

probe_df = pd.DataFrame(probe_results)
print("")
print("Sonda Linear Independente (treinada em train, avaliada em test):")
print(probe_df.to_string(index=False))



Sonda Linear Independente (treinada em train, avaliada em test):
     Model  Linear_Probe_Test_Acc  Linear_Probe_Macro_F1
  ResNet50                  0.015               0.005997
ResNet50_F                  0.020               0.005417


In [7]:
# 7. Projeção Qualitativa UMAP 2D (Figura 2)
umap_feats = embeddings_cache["convnext_noise_species"]
umap_y = labels_cache["convnext_noise_species"]
umap_df = export_umap_projection(
    features=umap_feats[:100],
    labels=umap_y[:100],
    output_path=output_dir / "umap_projection_convnext_f.csv",
    random_state=42,
)
print("")
print("[UMAP] Projeção UMAP 2D gerada com sucesso.")
print("AVISO CIENTÍFICO (AD-012): A projeção UMAP 2D destina-se estritamente à visualização qualitativa")
print("e NÃO demonstra distância filogenética ou separação formal de clusters.")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



[UMAP] Projeção UMAP 2D gerada com sucesso.
AVISO CIENTÍFICO (AD-012): A projeção UMAP 2D destina-se estritamente à visualização qualitativa
e NÃO demonstra distância filogenética ou separação formal de clusters.


In [8]:
# 8. Emissão do Manifesto de Execução Rastreável
manifest = write_notebook_manifest(
    notebook_id="06_ictv_latent_space_and_hierarchy",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    output_json_path=output_dir / "manifest.json",
    input_files=[str(table5_path), str(table6_path)],
    metrics_summary={
        "models_evaluated": len(models_to_evaluate),
        "table5_entries": len(table5_df),
        "table6_entries": len(table6_df),
        "umap_exported": True,
    },
)
print("\nManifesto de Execução salvo com sucesso:")
print(json.dumps(manifest.metrics_summary, indent=2))



Manifesto de Execução salvo com sucesso:
{
  "models_evaluated": 5,
  "table5_entries": 15,
  "table6_entries": 10,
  "umap_exported": true
}
